# 03 — Outer Vector Add（外积向量加法）

## 背景

在深度学习中，很多操作天然具有**二维结构**：批次 × 序列长度、行 × 列等。为了充分并行，需要在两个维度上同时展开计算，这就要求掌握**二维 GPU 内核网格**。

外积向量加法是 2D 广播的最基础形式，也是理解矩阵运算（如位置编码加法、偏置广播）的基础。

## 问题定义

给定 1D 张量 `A [N]` 和 `B [M]`，计算 2D 张量 `C [N, M]`：

$$C[i, j] = A[i] + B[j], \quad i \in [0, N),\ j \in [0, M)$$

等价写法：`C = A[:, None] + B[None, :]`

**并行策略**：启动二维网格 `(N//BN, M//BM)`，每个 Block 负责输出矩阵的 `BN × BM` 子块。

In [ ]:
import tilelang
import tilelang.language as T
import triton
import triton.language as tl
import torch

tilelang.disable_cache()
print("TileLang:", tilelang.__version__)
print("Triton:  ", triton.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")

## PyTorch 参考实现

In [ ]:
N, M = 8192, 4096
A = torch.randn(N, dtype=torch.float16, device="cuda")
B = torch.randn(M, dtype=torch.float16, device="cuda")

def ref_outer_add(A, B):
    return A[:, None] + B[None, :]   # [N, M]

C_ref = ref_outer_add(A, B)
print(f"A: {A.shape}, B: {B.shape} → C: {C_ref.shape}")

## Triton 实现

Triton 使用 `tl.program_id(0/1)` 获取二维网格坐标，通过 `rows[:, None]` 和 `cols[None, :]` 构造 2D 下标矩阵，一次 `tl.store` 写出整个 `BLOCK_N × BLOCK_M` 子块。

In [ ]:
@triton.jit
def _outer_add_kernel(A_ptr, B_ptr, C_ptr, N, M,
                      BN: tl.constexpr, BM: tl.constexpr):
    pid_n = tl.program_id(0)
    pid_m = tl.program_id(1)
    rows  = pid_n * BN + tl.arange(0, BN)
    cols  = pid_m * BM + tl.arange(0, BM)
    mask  = (rows[:, None] < N) & (cols[None, :] < M)
    a = tl.load(A_ptr + rows, mask=rows < N, other=0.0)
    b = tl.load(B_ptr + cols, mask=cols < M, other=0.0)
    c = a[:, None] + b[None, :]
    tl.store(C_ptr + rows[:, None] * M + cols[None, :], c, mask=mask)

# ── GPU 自适应 Triton 分块 ─────────────────────────────────────────────────────
# gfx1100 / gfx1201: BN=64, BM=64 — 均衡 2D tile，独立显存访问快
# gfx1151 (iGPU, 统一内存): BN=1, BM=2048 — 每个 program 处理 1 行
#   BN=64 导致跨行跨步写 (stride=M=8KB) → 统一内存 cache miss
#   BN=1: 所有线程写同一行连续列 → 完全 coalesced
arch_tri = torch.cuda.get_device_properties(0).gcnArchName
if arch_tri.startswith("gfx1151"):
    BN_t, BM_t = 1, 2048
else:
    BN_t, BM_t = 64, 64

def triton_outer_add(A, B):
    C = torch.empty(N, M, dtype=A.dtype, device=A.device)
    grid = (triton.cdiv(N, BN_t), triton.cdiv(M, BM_t))
    _outer_add_kernel[grid](A, B, C, N, M, BN=BN_t, BM=BM_t)
    return C

C_tri = triton_outer_add(A, B)
print("Triton correctness:", "✅ PASSED" if torch.allclose(C_ref, C_tri, atol=1e-2) else "❌ FAILED")

## TileLang 实现

TileLang 用 `T.Kernel(D0, D1)` 声明二维网格，`T.Parallel(BN, BM)` 展开二维并行循环，语法上比 Triton 更接近 Python。

In [ ]:
# ── GPU 自适应配置 ────────────────────────────────────────────────────────────
# gfx1100 (RX 7900 XTX): BN=256, BM=64, threads=256（+121% vs PyTorch）
# gfx1201 (R9700): 同样非对称分块，适配 RDNA4 L1 缓存
# gfx1151 (Radeon 8060S): RDNA3.5 iGPU，统一内存
#   关键问题：T.Parallel(BN, BM) 大 BN 导致跨行跨步写（stride = M = 8KB）→ cache miss
#   解决：BN=1，每块处理 1 行，所有线程写同一行连续列，完全 coalesced
#   Grid: (N, M//BM)，结果：+7% vs PyTorch，+85% vs Triton
arch = torch.cuda.get_device_properties(0).gcnArchName

if arch.startswith("gfx1151"):
    @tilelang.jit(pass_configs={
        tilelang.PassConfigKey.TL_DISABLE_WARP_SPECIALIZED: True,
        tilelang.PassConfigKey.TL_DISABLE_TMA_LOWER: True,
    })
    def tl_outer_add(A, B, BLOCK_M: int):
        N, M = T.const("N, M")
        dtype = T.float16
        A: T.Tensor((N,), dtype);  B: T.Tensor((M,), dtype)
        C = T.empty((N, M), dtype)
        with T.Kernel(N, M // BLOCK_M, threads=256) as (row, pid_m):
            for j in T.Parallel(BLOCK_M):
                C[row, pid_m * BLOCK_M + j] = A[row] + B[pid_m * BLOCK_M + j]
        return C

    k = tl_outer_add.compile(N=N, M=M, BLOCK_M=4096)

else:
    BN, BM = 256, 64

    @tilelang.jit(pass_configs={
        tilelang.PassConfigKey.TL_DISABLE_WARP_SPECIALIZED: True,
        tilelang.PassConfigKey.TL_DISABLE_TMA_LOWER: True,
    })
    def tl_outer_add(A, B, BLOCK_N: int, BLOCK_M: int):
        N, M = T.const("N, M")
        dtype = T.float16
        A: T.Tensor((N,), dtype);  B: T.Tensor((M,), dtype)
        C = T.empty((N, M), dtype)
        with T.Kernel(N // BLOCK_N, M // BLOCK_M, threads=256) as (pid_n, pid_m):
            for i, j in T.Parallel(BLOCK_N, BLOCK_M):
                row = pid_n * BLOCK_N + i
                col = pid_m * BLOCK_M + j
                C[row, col] = A[row] + B[col]
        return C

    k = tl_outer_add.compile(N=N, M=M, BLOCK_N=BN, BLOCK_M=BM)

C_tl = k(A.clone(), B.clone())
print("TileLang 正确性:", "✅ PASSED" if torch.allclose(C_ref, C_tl, atol=1e-2) else "❌ FAILED")

## 性能对比

In [ ]:
WARMUP, REPEAT = 20, 200

def bench(fn, args, warmup=WARMUP, repeat=REPEAT):
    for _ in range(warmup): fn(*args)
    s = torch.cuda.Event(enable_timing=True)
    e = torch.cuda.Event(enable_timing=True)
    torch.cuda.synchronize(); s.record()
    for _ in range(repeat): fn(*args)
    e.record(); torch.cuda.synchronize()
    return s.elapsed_time(e) / repeat

results = {
    "PyTorch (broadcast)": bench(ref_outer_add,    [A, B]),
    "Triton (2D grid)":    bench(triton_outer_add, [A, B]),
    "TileLang (2D grid)":  bench(k,               [A, B]),
}

bytes_rw = N * 2 + M * 2 + N * M * 2   # float16
pt_ms  = results["PyTorch (broadcast)"]
tri_ms = results["Triton (2D grid)"]

header = f"{'实现':<22}  {'延迟 (ms)':>10}  {'带宽 (TB/s)':>12}  {'vs PyTorch':>10}  {'vs Triton':>10}"
sep    = "-" * len(header)
print(sep)
print(header)
print(sep)
for name, ms in results.items():
    bw     = bytes_rw / ms / 1e9
    vs_pt  = f"{pt_ms/ms:+.1%}"
    vs_tri = f"{tri_ms/ms:+.1%}"
    print(f"{name:<22}  {ms:>10.4f}  {bw:>12.2f}  {vs_pt:>10}  {vs_tri:>10}")
print(sep)
